# TabM — Parameter-Efficient Ensemble MLP

**Paper**: [TabM: Advancing Tabular Deep Learning with Parameter-Efficient Ensembling](https://arxiv.org/abs/2410.24210) (Gorishniy et al., ICLR 2025)  
**Library**: `tabm` (official Yandex package)

TabM trains `k` MLP submodels in parallel with shared backbone weights (BatchEnsemble-style).  
Key advantages over a single MLP:
- Built-in ensembling at minimal parameter overhead
- Ensemble-level early stopping (monitor averaged OOF AUC)
- Typically beats FT-Transformer on most benchmarks

**Baseline to beat**: CatBoost default OOF AUC = 0.95540

In [1]:
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tabm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'tabm version: {tabm.__version__ if hasattr(tabm, "__version__") else "installed"}')

PyTorch 2.0.1+cu118  |  Device: cuda
GPU: NVIDIA GeForce RTX 3060
tabm version: 0.0.3


## Data

In [2]:
KAGGLE_DATA = Path('/kaggle/input/playground-series-s6e2')
LOCAL_DATA  = Path('data')
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(pd.read_csv(DATA_DIR / 'train.csv'))
test  = prep(pd.read_csv(DATA_DIR / 'test.csv'))
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']

y = train['heart_disease'].values.astype(np.float32)
print(f'Train: {train.shape}  Test: {test.shape}')
print(f'Num features: {NUM_FEATURES}')
print(f'Cat features: {CAT_FEATURES}')

Train: (630000, 15)  Test: (270000, 14)
Num features: ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
Cat features: ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results', 'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']


In [3]:
scaler = StandardScaler()
enc    = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_num      = scaler.fit_transform(train[NUM_FEATURES].values).astype(np.float32)
X_num_test = scaler.transform(test[NUM_FEATURES].values).astype(np.float32)

X_cat      = enc.fit_transform(train[CAT_FEATURES].values).astype(np.int64)
X_cat_test = enc.transform(test[CAT_FEATURES].values).astype(np.int64)

# +1 to cardinality to absorb any unseen category
CARDINALITIES = [int(len(cats)) + 1 for cats in enc.categories_]

# Clip cat indices to valid range
for i, card in enumerate(CARDINALITIES):
    X_cat[:, i]      = np.clip(X_cat[:, i],      0, card - 1)
    X_cat_test[:, i] = np.clip(X_cat_test[:, i], 0, card - 1)

print(f'X_num: {X_num.shape}  X_cat: {X_cat.shape}')
print(f'Cardinalities: {dict(zip(CAT_FEATURES, CARDINALITIES))}')

X_num: (630000, 5)  X_cat: (630000, 8)
Cardinalities: {'sex': 3, 'chest_pain_type': 5, 'fbs_over_120': 3, 'ekg_results': 4, 'exercise_angina': 3, 'slope_of_st': 4, 'number_of_vessels_fluro': 5, 'thallium': 4}


## Dataset / DataLoader

In [4]:
class TabularDataset(Dataset):
    def __init__(self, x_num, x_cat, y=None):
        self.x_num = torch.from_numpy(x_num)
        self.x_cat = torch.from_numpy(x_cat)
        self.y     = torch.from_numpy(y) if y is not None else None

    def __len__(self): return len(self.x_num)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.x_num[idx], self.x_cat[idx], self.y[idx]
        return self.x_num[idx], self.x_cat[idx]


def make_loader(x_num, x_cat, y=None, batch_size=1024, shuffle=False):
    ds = TabularDataset(x_num, x_cat, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      pin_memory=(device.type == 'cuda'), num_workers=4)

## TabM Model

`TabM.make()` creates a TabM with `k=32` parallel MLP submodels (default).  
Output shape: `(batch_size, k, d_out)`.  
Training: compute BCE for each of the k outputs, average.  
Prediction: sigmoid → average over k → scalar probability.

In [5]:
def make_model():
    """TabM with default hyperparameters: k=32, n_blocks=3, d_block=512, dropout=0.1."""
    model = tabm.TabM.make(
        n_num_features   = len(NUM_FEATURES),
        cat_cardinalities= CARDINALITIES,
        d_out            = 1,
        # defaults: k=32, n_blocks=3, d_block=512, dropout=0.1, arch_type='tabm'
    )
    return model.to(device)


# Sanity check
_m = make_model()
n_params = sum(p.numel() for p in _m.parameters())
print(f'TabM params: {n_params:,}  ({n_params/1e6:.2f}M)')

# Forward shape check
_xn = torch.zeros(4, len(NUM_FEATURES)).to(device)
_xc = torch.zeros(4, len(CAT_FEATURES), dtype=torch.int64).to(device)
_out = _m(_xn, _xc)
print(f'Output shape: {_out.shape}  (batch=4, k=32, d_out=1)')
del _m, _xn, _xc, _out

TabM params: 691,360  (0.69M)
Output shape: torch.Size([4, 32, 1])  (batch=4, k=32, d_out=1)


## Training Functions

In [6]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, n = 0., 0
    for x_num, x_cat, y_batch in loader:
        x_num    = x_num.to(device)
        x_cat    = x_cat.to(device)
        y_batch  = y_batch.to(device)

        optimizer.zero_grad()
        logits   = model(x_num, x_cat).squeeze(-1)          # (B, k)
        y_expand = y_batch.unsqueeze(1).expand_as(logits)   # (B, k)
        loss     = F.binary_cross_entropy_with_logits(logits, y_expand)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * len(y_batch)
        n          += len(y_batch)
    return total_loss / n


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    preds = []
    for batch in loader:
        x_num, x_cat = batch[0].to(device), batch[1].to(device)
        logits = model(x_num, x_cat).squeeze(-1)  # (B, k)
        proba  = torch.sigmoid(logits).mean(dim=1) # (B,)  — average over k
        preds.append(proba.cpu().numpy())
    return np.concatenate(preds)


def train_fold(
    x_num_tr, x_cat_tr, y_tr,
    x_num_val, x_cat_val, y_val,
    batch_size=1024,
    max_epochs=100,
    patience=15,
    lr=1e-4,
    weight_decay=1e-5,
):
    tr_loader  = make_loader(x_num_tr,  x_cat_tr,  y_tr,  batch_size, shuffle=True)
    val_loader = make_loader(x_num_val, x_cat_val, y_val, batch_size * 4)

    model     = make_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr,
        steps_per_epoch=len(tr_loader),
        epochs=max_epochs,
    )

    best_auc, best_state, no_improve = 0., None, 0
    for epoch in range(1, max_epochs + 1):
        train_loss = train_epoch(model, tr_loader, optimizer, scheduler)
        val_preds  = predict_proba(model, val_loader)
        val_auc    = roc_auc_score(y_val, val_preds)

        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 10 == 0 or no_improve >= patience:
            print(f'  Epoch {epoch:3d} | loss={train_loss:.5f} | val_auc={val_auc:.5f}'
                  f' | best={best_auc:.5f} | no_imp={no_improve}')

        if no_improve >= patience:
            print(f'  Early stopping at epoch {epoch}')
            break

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    return model, best_auc

## 5-Fold Cross-Validation

In [7]:
BASELINE_AUC = 0.95540  # CatBoost default seed-42

N_SPLITS   = 5
BATCH_SIZE = 1024
MAX_EPOCHS = 100
PATIENCE   = 15
LR         = 1e-4

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof        = np.zeros(len(y))
test_folds = np.zeros((N_SPLITS, len(test)))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_num, y)):
    print(f'\n=== Fold {fold + 1}/{N_SPLITS} ===')

    test_loader = make_loader(X_num_test, X_cat_test, batch_size=BATCH_SIZE * 4)

    model, best_auc = train_fold(
        X_num[tr_idx],  X_cat[tr_idx],  y[tr_idx],
        X_num[val_idx], X_cat[val_idx], y[val_idx],
        batch_size  = BATCH_SIZE,
        max_epochs  = MAX_EPOCHS,
        patience    = PATIENCE,
        lr          = LR,
    )

    val_loader = make_loader(X_num[val_idx], X_cat[val_idx], batch_size=BATCH_SIZE * 4)

    oof[val_idx]      = predict_proba(model, val_loader)
    test_folds[fold]  = predict_proba(model, test_loader)

    print(f'Fold {fold + 1} AUC: {roc_auc_score(y[val_idx], oof[val_idx]):.5f}')

cv_auc     = roc_auc_score(y, oof)
test_preds = test_folds.mean(axis=0)

print(f'\nTabM OOF AUC:      {cv_auc:.5f}')
print(f'CatBoost baseline: {BASELINE_AUC:.5f}')
print(f'Delta:             {cv_auc - BASELINE_AUC:+.5f}')


=== Fold 1/5 ===
  Epoch  10 | loss=0.27985 | val_auc=0.95328 | best=0.95328 | no_imp=0
  Epoch  20 | loss=0.27675 | val_auc=0.95351 | best=0.95351 | no_imp=0
  Epoch  30 | loss=0.27552 | val_auc=0.95362 | best=0.95362 | no_imp=1
  Epoch  40 | loss=0.27488 | val_auc=0.95369 | best=0.95369 | no_imp=0
  Epoch  50 | loss=0.27448 | val_auc=0.95373 | best=0.95373 | no_imp=0
  Epoch  60 | loss=0.27421 | val_auc=0.95370 | best=0.95373 | no_imp=1
  Epoch  70 | loss=0.27397 | val_auc=0.95375 | best=0.95375 | no_imp=0
  Epoch  80 | loss=0.27382 | val_auc=0.95374 | best=0.95375 | no_imp=8
  Epoch  87 | loss=0.27375 | val_auc=0.95374 | best=0.95375 | no_imp=15
  Early stopping at epoch 87
Fold 1 AUC: 0.95375

=== Fold 2/5 ===
  Epoch  10 | loss=0.27906 | val_auc=0.95235 | best=0.95235 | no_imp=0
  Epoch  20 | loss=0.27612 | val_auc=0.95270 | best=0.95270 | no_imp=0
  Epoch  30 | loss=0.27495 | val_auc=0.95271 | best=0.95281 | no_imp=3
  Epoch  40 | loss=0.27425 | val_auc=0.95284 | best=0.95287 | 

## Save OOF + Submit

In [8]:
np.save('submissions/oof_tabm.npy',  oof)
np.save('submissions/test_tabm.npy', test_preds)

label = f'tabm_k32_d512_b3'
fname = f'submissions/{label}.csv'
sub   = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)
print(f'Saved: {fname}')

desc = f'{label} | cv_auc={cv_auc:.4f}'

ON_KAGGLE = Path('/kaggle/working').exists()
if ON_KAGGLE:
    print(f'Submit: kaggle competitions submit -c playground-series-s6e2 -f {fname} -m "{desc}"')
else:
    r = subprocess.run(
        ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
         '-f', fname, '-m', desc],
        capture_output=True, text=True
    )
    status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
    print(f'{label}: {status}')
    print(f'desc: {desc}')

Saved: submissions/tabm_k32_d512_b3.csv
tabm_k32_d512_b3: submitted
desc: tabm_k32_d512_b3 | cv_auc=0.9535


## SLSQP Ensemble with GBTs

Optimize weights over CatBoost + LGBM + XGB + TabM OOF predictions.

In [9]:
from scipy.optimize import minimize

oof_files  = {
    'catboost': 'submissions/oof_cat.npy',
    'lgbm':     'submissions/oof_lgb.npy',
    'xgb':      'submissions/oof_xgb.npy',
    'tabm':     'submissions/oof_tabm.npy',
}
test_files = {
    'catboost': 'submissions/test_cat.npy',
    'lgbm':     'submissions/test_lgb.npy',
    'xgb':      'submissions/test_xgb.npy',
    'tabm':     'submissions/test_tabm.npy',
}

available = {k: Path(v).exists() for k, v in oof_files.items()}
print('OOF files available:', available)

if all(available.values()):
    oofs  = {k: np.load(v) for k, v in oof_files.items()}
    tests = {k: np.load(v) for k, v in test_files.items()}

    names    = list(oofs.keys())
    oof_list = [oofs[n] for n in names]

    def neg_auc(w):
        return -roc_auc_score(y, sum(wi * o for wi, o in zip(w, oof_list)))

    res = minimize(
        neg_auc, x0=[1/len(names)] * len(names),
        method='SLSQP',
        bounds=[(0, 1)] * len(names),
        constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1},
    )
    opt_auc = -res.fun
    print('\nSLSQP ensemble weights:')
    for n, w in zip(names, res.x):
        print(f'  {n:<12} {w:.4f}')
    print(f'  OOF AUC = {opt_auc:.5f}  (vs CatBoost {BASELINE_AUC:.5f})')

    if opt_auc > BASELINE_AUC:
        pred   = sum(w * tests[n] for w, n in zip(res.x, names))
        fname2 = 'submissions/ensemble_gbt_tabm.csv'
        desc2  = f'ensemble_gbt_tabm | cv_auc={opt_auc:.4f}'
        sub2   = ss.copy()
        sub2['Heart Disease'] = pred
        sub2.to_csv(fname2, index=False)
        if not ON_KAGGLE:
            r2 = subprocess.run(
                ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
                 '-f', fname2, '-m', desc2],
                capture_output=True, text=True
            )
            print(f'ensemble: {"submitted" if "successfully" in r2.stdout.lower() else r2.stdout[:80]}')
    else:
        print('TabM did not improve ensemble — skipping ensemble submission.')
else:
    print('Run s6e2-gradient-boosting.ipynb first to generate GBT OOF files.')

OOF files available: {'catboost': True, 'lgbm': True, 'xgb': True, 'tabm': True}

SLSQP ensemble weights:
  catboost     0.2535
  lgbm         0.2480
  xgb          0.2516
  tabm         0.2470
  OOF AUC = 0.95518  (vs CatBoost 0.95540)
TabM did not improve ensemble — skipping ensemble submission.
